# LangGraph 01 — OpenAI: Chatbot có memory và Tool-calling Agent

In [1]:
import os
from getpass import getpass

def ensure_secret(name: str):
    if os.getenv(name):
        return
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = getpass(f"Nhập {name}: ")
    if not value:
        raise ValueError(f"Thiếu {name}")
    os.environ[name] = value

ensure_secret("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
print("Model:", OPENAI_MODEL)

Model: gpt-4.1-mini


## 1. Chatbot có short-term memory

`MessagesState` đã định nghĩa field `messages` cùng reducer để nối message mới vào lịch sử. Node chỉ cần trả message mới; không cần tự nối list.

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)

def tutor(state: MessagesState):
    # Code
    system = SystemMessage(content="Bạn là trợ giảng về Langgraph. Hãy trả lời các câu hỏi ngắn gọn và dễ hiểu.")
    response = llm.invoke([system, *state["messages"]])
    return {"messages": [response]}

In [5]:
# StateGraph
builder = StateGraph(MessagesState)

# add_node
builder.add_node("tutor", tutor)

# add_edge
builder.add_edge(START, "tutor")
builder.add_edge("tutor", END)

# Chatbot Memory
chatbot = builder.compile(checkpointer=InMemorySaver())

In [6]:
# Config thread_id
config = {"thread_id": "langgraph_tutor"}

first = chatbot.invoke(
    {"messages": [HumanMessage(content="Hãy nhớ tên tôi là An. StateGraph là gì?")]},
    config=config,
)
print("AI:", first["messages"][-1].content)

second = chatbot.invoke(
    {"messages": [HumanMessage(content="Tên tôi là gì?")]},
    config=config,
)
print("AI:", second["messages"][-1].content)

AI: Chào An! StateGraph là một loại đồ thị dùng để mô tả các trạng thái khác nhau của một hệ thống và các chuyển đổi giữa các trạng thái đó. Nó giúp hiểu và phân tích cách hệ thống thay đổi theo thời gian hoặc theo các sự kiện.
AI: Tên bạn là An.


## 2. Tool-calling agent tự xây bằng graph

Ta dùng hai tool deterministic. LLM quyết định có gọi tool hay không; `ToolNode` thực thi tool; `tools_condition` route về `END` nếu không còn tool call.

In [7]:
import ast
import operator
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

In [9]:
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}

def _safe_number(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_number(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_number(node.left), _safe_number(node.right))
    raise ValueError("Chỉ chấp nhận số và các phép +, -, *, /, **")

@tool
def calculator(expression: str) -> str:
    '''Tính một biểu thức số học an toàn, ví dụ '(120 * 3) / 4'.'''
    tree = ast.parse(expression, mode="eval")
    return str(_safe_number(tree.body))

@tool
def course_policy(topic: str) -> str:
    '''Tra cứu chính sách giả lập theo chủ đề attendance, homework hoặc support.'''
    policies = {
        "attendance": "Học viên cần tham dự ít nhất 80% số buổi.",
        "homework": "Bài tập nộp trước 23:59 Chủ nhật.",
        "support": "Hỗ trợ kỹ thuật trong 24 giờ làm việc.",
    }
    return policies.get(topic.lower(), "Không tìm thấy chính sách cho chủ đề này.")

In [10]:
# Tools
tools = [calculator, course_policy]
# Agent LLM
agent_llm = llm.bind_tools(tools)

In [11]:
def assistant(state: MessagesState):
    system = SystemMessage(
        content="Bạn là trợ giảng. Dùng tool khi cần tính toán hoặc tra chính sách. Trả lời tiếng Việt."
    )
    return {"messages": [agent_llm.invoke([system, *state["messages"]])]}

In [12]:
# StateGraph
agent_builder = StateGraph(MessagesState)

# add_node: assistant
agent_builder.add_node("assistant", assistant)

# add_node: tools
agent_builder.add_node("tools", ToolNode(tools))

# add_edge: START --> assistant
agent_builder.add_edge(START, "assistant")

# add_conditional_edges: choose appropriate tools
agent_builder.add_conditional_edges(
    "assistant", tools_condition
)

# add_edge: tools --> assistant
agent_builder.add_edge("tools", "assistant")
agent_builder.add_edge("assistant", END)

# Chat Memory
agent = agent_builder.compile(checkpointer=InMemorySaver())

print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	assistant(assistant)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> assistant;
	assistant -.-> __end__;
	assistant -.-> tools;
	tools --> assistant;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [13]:
agent_config = {"configurable": {"thread_id": "demo-agent-openai"}}
question = "Lớp có 120 học viên, chia đều 4 nhóm thì mỗi nhóm bao nhiêu người? Hạn nộp homework là khi nào?"

result = agent.invoke({"messages": [HumanMessage(content=question)]}, config=agent_config)
for message in result["messages"]:
    print(f"{message.type:>7} | {message.content or message.additional_kwargs}")

  human | Lớp có 120 học viên, chia đều 4 nhóm thì mỗi nhóm bao nhiêu người? Hạn nộp homework là khi nào?
     ai | {'refusal': None}
   tool | 30.0
   tool | Bài tập nộp trước 23:59 Chủ nhật.
     ai | Lớp có 120 học viên, chia đều 4 nhóm thì mỗi nhóm có 30 người. Hạn nộp bài tập (homework) là trước 23:59 Chủ nhật. Bạn cần hỗ trợ gì thêm không?
